# MRC-Ox Go/No-Go/Conflict (excluded)

Dataset key: `mrc_ox_gonogo`

Source: MRC-Ox data portal requires registration/login before download; no anonymous bundle can be materialized from this repository.

This notebook documents the exclusion and is not counted as a plotted Week 19 data source. The real downloadable Go/No-Go source plotted in Week 19 is `openneuro_gonogo_ds002680`.


In [ ]:
import Pkg

function find_repo_root()
    candidates = unique(normpath.([
        pwd(),
        joinpath(pwd(), ".."),
        joinpath(pwd(), "..", ".."),
        joinpath(pwd(), "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from pwd=$(pwd()).")
end

const REPO_ROOT = find_repo_root()
const NOTEBOOK_DIR = joinpath(REPO_ROOT, "notebooks", "week_19", "data_sources")
const DATASETS_ROOT = joinpath(REPO_ROOT, "notebooks", "datasets")
const WEEK19_DOWNLOADS = joinpath(REPO_ROOT, "notebooks", "week_19", "downloads")
const PYTHON = begin
    venv_python = joinpath(REPO_ROOT, ".venv_8bit", "bin", "python")
    isfile(venv_python) ? venv_python : "python"
end

Pkg.activate(joinpath(REPO_ROOT, "notebooks", "model_test"))

using CairoMakie
using CSV
using DataFrames
using HDF5
using JSON3
using Printf
using Statistics

include(joinpath(REPO_ROOT, "notebooks", "week_15", "try_new_data_helpers.jl"))
using .Week15TryNewData

mkpath(WEEK19_DOWNLOADS)
RNG_SEED = Int(mod(time_ns(), UInt64(typemax(Int))))
println("Repo root: ", REPO_ROOT)
println("Python: ", PYTHON)
println("RNG seed: ", RNG_SEED)


In [ ]:
function bundle_files(dataset_key::AbstractString)
    dir = joinpath(DATASETS_ROOT, dataset_key)
    return (
        dir = dir,
        h5 = joinpath(dir, "epochs.hdf5"),
        events = joinpath(dir, "events.csv"),
        metadata = joinpath(dir, "metadata.json"),
    )
end

function standard_bundle_ready(dataset_key::AbstractString)
    files = bundle_files(dataset_key)
    all(isfile, [files.h5, files.events, files.metadata]) || return false
    return h5open(files.h5, "r") do f
        if haskey(f, "subjects")
            return length(keys(f["subjects"])) > 0
        end
        return true
    end
end

function maybe_run_import!(cmd::Cmd; force::Bool = false)
    if RUN_IMPORT || force
        run(cmd)
    else
        @info "RUN_IMPORT=false; not running importer. Set RUN_IMPORT=true in this notebook to rebuild/download."
    end
end

const DATASET_KEY = "mrc_ox_gonogo"
const SOURCE_NOTE = "Excluded: MRC-Ox requires registration/login, so no anonymous source bundle can be downloaded or plotted here. Use openneuro_gonogo_ds002680 as the real plotted Go/No-Go source."
const RUN_IMPORT = false
const PREPARE_SCRIPT = joinpath(REPO_ROOT, "scripts", "prepare_sigmoid_public_datasets.py")
const IMPORT_ARGS = String[
    "--output-root",
    DATASETS_ROOT,
    "--datasets",
    "mrc_ox_gonogo",
]

@assert isfile(PREPARE_SCRIPT) "Missing importer: $PREPARE_SCRIPT"

files = bundle_files(DATASET_KEY)
if RUN_IMPORT
    println("Forcing rebuild: ", files.dir)
    maybe_run_import!(Cmd(vcat([PYTHON, PREPARE_SCRIPT], IMPORT_ARGS)); force = true)
elseif standard_bundle_ready(DATASET_KEY)
    println("Bundle already ready: ", files.dir)
else
    println("Bundle missing/incomplete: ", files.dir)
    maybe_run_import!(Cmd(vcat([PYTHON, PREPARE_SCRIPT], IMPORT_ARGS)))
end


In [ ]:
files = bundle_files(DATASET_KEY)
materialized = standard_bundle_ready(DATASET_KEY)

mrc_status = DataFrame(
    dataset_key = [DATASET_KEY],
    materialized = [materialized],
    plotted = [false],
    counted_in_week19 = [false],
    replacement_dataset_key = ["openneuro_gonogo_ds002680"],
    reason = [SOURCE_NOTE],
    bundle_dir = [files.dir],
)
display(mrc_status)

@assert !materialized "MRC-Ox was manually materialized; add a dedicated source-specific import and plotting path before counting it."
println("No MRC-Ox preview generated; excluded from Week 19 plotted-source count.")
